# 03 · Filter & Rank — the shared enzyme filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 24** you filter on the **enzyme** cutoffs, where `cat_geom` is the
**coordination**-geometry RMSD and `plddt_cat` is the **site** confidence — the coordination geometry
is the decisive metric.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
Improvements here are pull-requested back to `shared/` for the whole cohort — do not silently fork it.
We reuse `design_type="enzyme"`: for a cofactor site, `cat_geom` maps to **coordination**-geometry and
`plddt_cat` to **site** confidence (pLDDT at the coordinating residues).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("enzyme cutoffs:", fp.DEFAULT_CUTOFFS["enzyme"])
print("  (cat_geom -> coordination-geometry RMSD; plddt_cat -> site pLDDT, for this project)")

## Build `fp.Design` objects (enzyme) from the campaign
Map each campaign row onto an `fp.Design`, carrying the metalloprotein-specific fields: `plddt`, the
**site** confidence into **`plddt_catalytic`**, `scrmsd`, and the **coordination**-geometry RMSD into
**`catalytic_geom_rmsd`**. The self-consistency layer (`self_consistency`) checks all of these against
the enzyme cutoffs (scrmsd ≤ 2.0, plddt ≥ 85, plddt_cat ≥ 90, cat_geom ≤ 0.5).

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence=str(r.get("sequence", "")),
        design_type="enzyme",
        plddt=float(r["plddt"]),
        plddt_catalytic=float(r["plddt_site"]),          # SITE confidence -> plddt_cat
        scrmsd=float(r["scrmsd"]),
        catalytic_geom_rmsd=float(r["coordination_geom_rmsd"]),  # COORDINATION geometry -> cat_geom
        md_rmsd=float(r["md_rmsd"]),
        extra={"scaffold_tool": r["scaffold_tool"], "cofactor": r["cofactor"], "synthetic": True},
    ))
print(len(designs), "enzyme Design objects built (from SYNTHETIC mock metrics)")
print("mapping: plddt_site -> plddt_catalytic; coordination_geom_rmsd -> catalytic_geom_rmsd")

## Run the pipeline (`design_type="enzyme"`) and report
`run_pipeline` applies the layers in order and returns a ranked DataFrame. We use layers 1+3+4
(self-consistency incl. coordination geometry, physics, and the short-MD dynamics layer); the
orthogonal layer (L2) needs a second predictor's scRMSD, which you add on Colab.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="enzyme", use_layers=(1, 3, 4))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj24")
top

## Survival-at-each-layer + coordination-geometry pass rate (honest accounting)
The **coordination-geometry layer is where most metalloprotein designs die** — expect the steepest
drop there (a clean bis-His geometry on opposite helices is hard to scaffold). Report the pass rate
explicitly; this is the headline benchmark for D3. **And remember:** passing it does **not** mean the
cofactor incorporates — only spectroscopy decides.

In [ ]:
import pandas as pd
print("layers_passed distribution:")
print(df_ranked["layers_passed"].value_counts().sort_index())

n = len(df_ranked)
cut = fp.DEFAULT_CUTOFFS["enzyme"]["cat_geom"]
n_geom = int((df_ranked["catalytic_geom_rmsd"] <= cut).sum())
print(f"\ncoordination-geometry preservation: {n_geom}/{n} "
      f"({100*n_geom/max(n,1):.1f}%) hold the motif < {cut} A  [SYNTHETIC demo numbers]")
print("REMINDER: coordination geometry != cofactor incorporation != function. Spectroscopy decides.")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="enzyme"`.
- [ ] Survival-at-each-layer figure (`results/proj24_survival.png`).
- [ ] Coordination-geometry preservation rate reported (the headline metric).
- [ ] Mapping assumptions written down (site pLDDT → `plddt_cat`; coordination RMSD → `cat_geom`).

**Next:** `04_validate.ipynb` — coordination-geometry preservation + cofactor/scheme comparison +
docking / site-pLDDT / MD figures + redox-tuning reasoning.